In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv('C:/Users/fedor/fluent_python/ab_data.csv')
df.head()

,user_id,group,converted,revenue,time_sec,clicks,sessions,device
0,A_000,A,0,0.00,42.6,4,2,mobile
1,A_001,A,1,133.77,71.1,2,2,mobile
2,A_002,A,0,0.00,73.3,6,1,mobile
3,A_003,A,0,0.00,47.6,3,1,mobile
4,A_004,A,0,0.00,48.9,0,5,desktop


In [3]:
# Когда ты проверяешь одну гипотезу при α = 0.05, вероятность ложноположительного = 5%. Это контролируемо.
# Когда проверяешь несколько гипотез — каждая с α = 0.05 — суммарная вероятность хотя бы одного ложноположительного растёт.
# P(хотя бы один ложноположительный) = 1−(1−α)^k (k - число тестов)

## Три семейства поправок

### Контроль FWER (Family-Wise Error Rate)

**Цель:** вероятность хотя бы одного ложноположительного ≤ α.

| Метод | Как работает | Когда |
| :--- | :--- | :--- |
| Бонферрони | α / k | самый строгий, простой |
| Холм | последовательный Бонферрони | менее консервативный |
| Šidák | 1 − (1−α)^(1/k) | чуть точнее Бонферрони |

### Контроль FDR (False Discovery Rate)

**Цель:** доля ложных среди отвергнутых ≤ q.

| Метод | Как работает | Когда |
| :--- | :--- | :--- |
| Benjamini-Hochberg (BH) | сортировка p-value + порог | мягче, для много гипотез |

### Без поправки

Если тесты независимы или их мало — иногда оставляют α = 0.05 на каждый, но явно оговаривают это в отчёте.

---

## Какую поправку брать в A/B-тесте

| Ситуация | Рекомендация |
| :--- | :--- |
| Мало метрик (2–5), критичное решение | Бонферрони или Холм |
| Много метрик (10+) | BH (FDR) |
| Одна метрика, но много сегментов | Холм или BH |
| Метрики заранее зафиксированы и независимы | можно без поправки, но с оговоркой |
| Exploratory анализ | BH для скрининга |

**Для нашего случая (4 метрики) → Холм или Бонферрони.**

In [6]:
import numpy as np
from statsmodels.stats.multitest import multipletests

p_values = {
    'converted': 0.3272,
    'time_sec':  0.0428,
    'clicks':    0.0070,
    'device':    0.0570,
}

names = list(p_values.keys())
p_arr = np.array(list(p_values.values()))

# 1) Bonferroni (самый строгий)
# Делим альфу на количество метрик, если меньше альфы, значит проходит тест
rej_b, p_adj_b, _, _ = multipletests(p_arr, alpha=0.05, method='bonferroni')

# 2) Holm (менее строгий)
# Для i-го (по порядку) сравнить с α / (k − i + 1).
# Остановиться на первом невыполнении
rej_h, p_adj_h, _, _ = multipletests(p_arr, alpha=0.05, method='holm')

# 3) Benjamini-Hochberg (FDR)
# Упорядочить p по возрастанию.
# Сравнить i-е p с i/k × q.
rej_bh, p_adj_bh, _, _ = multipletests(p_arr, alpha=0.05, method='fdr_bh')

# Сводка
import pandas as pd
summary = pd.DataFrame({
    'p_original':   p_arr,
    'p_bonferroni': p_adj_b.round(4),
    'p_holm':       p_adj_h.round(4),
    'p_bh':         p_adj_bh.round(4),
    'sig_bonf':     rej_b,
    'sig_holm':     rej_h,
    'sig_bh':       rej_bh,
}, index=names)
print(summary)

           p_original  p_bonferroni  p_holm    p_bh  sig_bonf  sig_holm  \
converted      0.3272        1.0000  0.3272  0.3272     False     False   
time_sec       0.0428        0.1712  0.1284  0.0760     False     False   
clicks         0.0070        0.0280  0.0280  0.0280      True      True   
device         0.0570        0.2280  0.1284  0.0760     False     False   

           sig_bh  
converted   False  
time_sec    False  
clicks       True  
device      False  


In [7]:

print("""
    
Мы проверили 4 метрики. Для контроля ложноположительных срабатываний применили поправку Холма. 
После поправки значимый эффект сохраняется только по метрике clicks (+34.5%, p_adj = 0.028). 
Метрика time_sec показала эффект до поправки (p = 0.043), но после коррекции на множественные сравнения теряет значимость (p_adj = 0.128).

""")



Мы проверили 4 метрики. Для контроля ложноположительных срабатываний применили поправку Холма. 
После поправки значимый эффект сохраняется только по метрике clicks (+34.5%, p_adj = 0.028). 
Метрика time_sec показала эффект до поправки (p = 0.043), но после коррекции на множественные сравнения теряет значимость (p_adj = 0.128).


